# Tests: `fasterai.export.onnx_exporter` (source `nbs/export/onnx_exporter.ipynb`)

In [ ]:
from fastcore.test import *
import warnings
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from fasterai.export.onnx_exporter import *
from fasterai.export.onnx_exporter import _has_package, _pt2e_translation_table
import fasterai.quantize.quantizer  # registers the pt2e q/dq operators used below

In [ ]:
from fastcore.test import *

# --- the translation table covers every q/dq operator the pt2e flow can emit ---
if _has_package("onnxscript"):
    _table = _pt2e_translation_table()
    _ops = torch.ops.quantized_decomposed
    for _overload in (_ops.quantize_per_tensor.default, _ops.dequantize_per_tensor.default,
                      _ops.quantize_per_channel.default, _ops.dequantize_per_channel.default):
        assert _overload in _table, f"missing translation for {_overload}"
    # per-tensor and per-channel have different signatures (`axis` only on the latter)
    import inspect
    test_eq(list(inspect.signature(_table[_ops.quantize_per_channel.default]).parameters)[3], 'axis')
    # the translations are plain functions: an `@onnxscript.script` wrapper would return a tuple
    assert all(inspect.isfunction(fn) for fn in _table.values())

# --- QDQStats is a plain, serializable record ---
test_eq(QDQStats(1, 2, 3, 4, 5).as_dict(),
        {'n_quantize': 1, 'n_dequantize': 2, 'n_per_channel': 3, 'n_nonzero_zero_point': 4,
         'n_unquantized_conv_add': 5})

In [ ]:
# --- qdq_stats reads every way a zero-point can be written into a graph ---
import tempfile

if _has_package("onnx"):
    import onnx
    from onnx import TensorProto, helper, numpy_helper

    def _tiny_qdq(path, zero_point=None, as_constant=False, per_channel=False, from_input=False):
        "Hand-built QuantizeLinear/DequantizeLinear pair with a controllable zero-point"
        scale = np.full(4, 0.05, np.float32) if per_channel else np.array(0.05, np.float32)
        initializers = [numpy_helper.from_array(scale, "scale")]
        inputs = [helper.make_tensor_value_info("x", TensorProto.FLOAT, [1, 4])]
        nodes, q_inputs = [], ["x", "scale"]
        if from_input:  # zero-point only known at runtime: unreadable from the graph
            inputs.append(helper.make_tensor_value_info("zp", TensorProto.INT8, []))
            q_inputs.append("zp")
        elif zero_point is not None:
            zp = np.full(4, zero_point, np.int8) if per_channel else np.array(zero_point, np.int8)
            if as_constant:
                nodes.append(helper.make_node("Constant", [], ["zp"],
                                              value=numpy_helper.from_array(zp, "zp_value")))
            else:
                initializers.append(numpy_helper.from_array(zp, "zp"))
            q_inputs.append("zp")
        attrs = {"axis": 1} if per_channel else {}
        nodes += [helper.make_node("QuantizeLinear", q_inputs, ["q"], **attrs),
                  helper.make_node("DequantizeLinear", ["q"] + q_inputs[1:], ["y"], **attrs)]
        graph = helper.make_graph(nodes, "qdq", inputs,
                                  [helper.make_tensor_value_info("y", TensorProto.FLOAT, [1, 4])],
                                  initializer=initializers)
        model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 18)])
        onnx.checker.check_model(model)
        onnx.save(model, str(path))
        return path

    with tempfile.TemporaryDirectory() as _tmp:
        _p = Path(_tmp)
        # zero-point stored as an initializer: the pin CAN fail
        test_eq(qdq_stats(_tiny_qdq(_p/'zero.onnx', zero_point=0)).as_dict(),
                {'n_quantize': 1, 'n_dequantize': 1, 'n_per_channel': 0, 'n_nonzero_zero_point': 0,
                 'n_unquantized_conv_add': 0})
        test_eq(qdq_stats(_tiny_qdq(_p/'nonzero.onnx', zero_point=7)).n_nonzero_zero_point, 2)
        # zero-point produced by a Constant node
        test_eq(qdq_stats(_tiny_qdq(_p/'const.onnx', zero_point=3, as_constant=True)).n_nonzero_zero_point, 2)
        # zero-point omitted altogether: implicitly zero
        test_eq(qdq_stats(_tiny_qdq(_p/'omitted.onnx')).n_nonzero_zero_point, 0)
        # per-channel scales are detected from the scale itself, not from the `axis` attribute
        test_eq(qdq_stats(_tiny_qdq(_p/'channel.onnx', zero_point=0, per_channel=True)).n_per_channel, 2)
        # a zero-point that is not a graph constant must raise, not be counted as zero
        with ExceptionExpected(ValueError, regex="zero_point"):
            qdq_stats(_tiny_qdq(_p/'runtime.onnx', from_input=True))

In [ ]:
# --- end to end: a pt2e-quantized model exported, inspected and verified ---
from fasterai.quantize.quantizer import Quantizer, _HAS_PT2E

if _HAS_PT2E and all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
    import onnx

    class _TinyConvNet(nn.Module):
        "Conv-BN-ReLU-Conv-Pool-Linear network, small enough to quantize in a test"
        def __init__(self, n_classes=10):
            super().__init__()
            self.features = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),
                                          nn.Conv2d(8, 16, 3, padding=1), nn.AdaptiveAvgPool2d(1))
            self.head = nn.Sequential(nn.Flatten(), nn.Linear(16, n_classes))
        def forward(self, x): return self.head(self.features(x))
        def spread_head(self, batch):
            "Center the classifier on the mean feature of `batch`, so its argmax follows the input"
            # An untrained head answers the same class to every input: its bias swamps the differences
            # between feature vectors. Reading the logits from the feature DEVIATIONS is what spreads a
            # batch over classes; the weights stay exactly as initialized, since rescaling them moves
            # every logit by the same factor and leaves the argmax where it was.
            with torch.no_grad():
                self.head[1].bias.copy_(-(self.head[1].weight @ self.features(batch).flatten(1).mean(0)))
            return self

    torch.manual_seed(0)
    _calib = [(torch.randn(4, 3, 16, 16), torch.randint(0, 10, (4,))) for _ in range(4)]
    _sample = torch.randn(4, 3, 16, 16)

    # `verify_qdq` compares ARGMAX, and an untrained net answers the same class to every input: its
    # agreement would read 1.0 against any graph at all. `spread_head` is what gives the agreements
    # below something to measure, and the spread is asserted before they are read.
    # Measured here (torch 2.9.1, onnxruntime CPU): over the 32 probes the reference spans 8 classes
    # (5 over the eight-probe slice) and agrees with its exported graph on 30 of 32 (0.938). 0.9 where
    # this check used to read 0.99 is deliberate — 0.938 is what a reference whose predictions vary
    # actually reads, because it sits near decision boundaries where ONNX Runtime's INT8 kernels round
    # differently than PyTorch's; on 32 probes 0.9 tolerates at most 3 disagreements. The negative
    # control — the same probe read against ANOTHER model's graph, where the agreement collapses —
    # lives with the pt2e fixtures in quantize/quantizer.ipynb and is not duplicated here.
    _MIN_CLASSES, _MIN_SLICE_CLASSES = 4, 2
    _MIN_AGREEMENT, _MIN_SLICE_AGREEMENT = 0.9, 0.75
    _qmodel = Quantizer(backend='pt2e').quantize(
        _TinyConvNet().eval().spread_head(torch.randn(64, 3, 16, 16)), _calib)

    with tempfile.TemporaryDirectory() as _tmp:
        _path = export_qdq(_qmodel, _sample, Path(_tmp)/'qdq.onnx')
        assert _path.exists()

        _stats = qdq_stats(_path)
        assert _stats.n_quantize > 0 and _stats.n_dequantize > 0, _stats
        assert _stats.n_per_channel > 0, "per-channel weights did not survive the export"
        test_eq(_stats.n_nonzero_zero_point, 0)  # portability: zero_point == 0 everywhere

        _static_batch = _sample.shape[0]  # the exported graph only accepts the batch it was traced on
        _spread_probe = torch.randn(32, 3, 16, 16)
        _n_batches = _spread_probe.shape[0] // _static_batch
        with torch.no_grad():
            _preds = torch.cat([_qmodel(_c).argmax(-1)
                                for _c in _spread_probe.split(_static_batch)]).tolist()
        assert len(set(_preds)) >= _MIN_CLASSES, ("the reference predictions must span classes, or "
                                                  f"the agreement below proves nothing: {_preds}")
        with warnings.catch_warnings(record=True) as _caught:
            warnings.simplefilter('always')
            _agreement = verify_qdq(_qmodel, _path, _spread_probe, n_batches=_n_batches)
        assert _agreement >= _MIN_AGREEMENT, f"ONNX and PyTorch disagree too often: {_agreement}"
        assert not any('vacuous' in str(w.message) for w in _caught), [str(w.message) for w in _caught]

        # custom tensor names reach the graph
        _named = export_qdq(_qmodel, _sample, Path(_tmp)/'named.onnx',
                            input_names=['images'], output_names=['logits'])
        _graph = onnx.load(str(_named)).graph
        test_eq([i.name for i in _graph.input], ['images'])
        test_eq([o.name for o in _graph.output], ['logits'])

        # dynamic_batch is announced as experimental, and says so again when it could not be honoured
        with warnings.catch_warnings(record=True) as _caught:
            warnings.simplefilter('always')
            export_qdq(_qmodel, _sample, Path(_tmp)/'dynamic.onnx', dynamic_batch=True)
        _messages = [str(w.message) for w in _caught if issubclass(w.category, UserWarning)]
        assert any('experimental' in m for m in _messages), _messages
        assert any('STATIC batch' in m for m in _messages), _messages

        # an empty sample is an error, never a silent 0.0 agreement
        with ExceptionExpected(ValueError, regex="empty"):
            verify_qdq(_qmodel, _path, torch.empty(0, 3, 16, 16))

        # `n_batches` must divide `sample`: an uneven last batch is exactly what a static
        # graph rejects (torch.chunk would turn 10 inputs into 3/3/3/1)
        with ExceptionExpected(ValueError, regex="equal batches"):
            verify_qdq(_qmodel, _path, torch.randn(10, 3, 16, 16), n_batches=4)
        # ...and a shorter probe read in two batches answers the same way — on a slice whose own
        # predictions still span classes, so this reading cannot be vacuous either. Its floor is
        # looser because eight probes move the agreement in steps of 0.125: 0.875 (7 of 8) here,
        # and _MIN_SLICE_AGREEMENT leaves room for one more boundary case to round the other way.
        _slice = _spread_probe[:2 * _static_batch]
        assert len(set(_preds[:2 * _static_batch])) >= _MIN_SLICE_CLASSES, _preds[:2 * _static_batch]
        _slice_agreement = verify_qdq(_qmodel, _path, _slice, n_batches=2)
        assert _slice_agreement >= _MIN_SLICE_AGREEMENT, \
            f"the short probe disagrees too often: {_slice_agreement}"